# Fine-tune ReactionT5 on a larger ORD sample (Model 1, Kaggle GPU)

Same `scripts/train_reactant_model_ord.py` as the Colab notebook (`colab/05_train_reactant_ord.ipynb`), pointed at a bigger training pool (~150,000 ORD reactions instead of 60,000) to test whether more data pushes accuracy past the v2 result (ORD exact_match 50.7%/top-5 74.3%).

**Before running:** in the notebook Settings panel (right sidebar) turn on **Internet** and **GPU accelerator** (T4x2). Kaggle's free GPU quota is **30 hours/week** (not per-day like Colab's free tier), which is why this is worth doing here instead of just extending the Colab runs.

**Fixed since the first 150k attempt:** that run (see `RESULTS.md` section 5) was accidentally confounded -- it launched without `--no-augment`, silently inheriting the script's default SMILES augmentation (prob=0.5), which independent testing (variant 4) already showed hurts accuracy despite a healthy-looking `eval_loss` curve. The training cell below now passes `--no-augment` explicitly, so this run is a clean "more data alone" test.

**Also fixed: actually using both GPUs.** Launching with plain `python` and 2 GPUs visible makes HF `Trainer` fall back to `torch.nn.DataParallel`, which adds overhead (Python-side gather/scatter across GPUs) without reliably speeding anything up -- confirmed by the 250k Kaggle run showing *lower* throughput than a single T4 on Colab, plus the tell-tale `"Was asked to gather along dimension 0, but all input tensors were scalars"` warning. The cell below now launches via `torchrun --nproc_per_node=2`, which makes Trainer use proper `DistributedDataParallel` instead -- each GPU runs its own process with no Python-level bottleneck, and the effective batch size doubles (16/GPU x 2 GPUs), roughly halving the number of steps needed per epoch on top of the per-step speedup.

**Data:** the 150k-reaction sample was built locally (`build_train_data_ord.py --pool-count 150000`, same seed/eval-exclusion logic as the 60k pool, so it's still leak-free against `data/v2_ord_eval_targets.json`) and must be uploaded as a **Kaggle Dataset** (two files: `reactants_train.jsonl`, `reactants_val.jsonl`), then added to this notebook as an input (`+ Add Input` in the right sidebar).

In [ ]:
import torch
print("CUDA available:", torch.cuda.is_available())
print("Device count:", torch.cuda.device_count())
for i in range(torch.cuda.device_count()):
    print(f"  Device {i}:", torch.cuda.get_device_name(i))
if torch.cuda.device_count() < 2:
    print("WARNING: fewer than 2 GPUs visible -- the torchrun --nproc_per_node=2 launch below expects 2.")

In [ ]:
import os

if not os.path.isdir("retro-planner"):
    !git clone https://github.com/oleh-kuzmenko/retro-planner.git
%cd retro-planner

In [ ]:
%pip install -q -e ".[local-models,indexing]"

**Input data.** Adjust the dataset slug below to match whatever you named the Kaggle Dataset you uploaded (visible under `/kaggle/input/` once added as an input).

In [ ]:
import os

train_file = "/kaggle/input/retro-planner-ord-150k/reactants_train.jsonl"  # @param {type:"string"}
val_file = "/kaggle/input/retro-planner-ord-150k/reactants_val.jsonl"  # @param {type:"string"}

assert os.path.exists(train_file), f"Not found: {train_file} -- did you add the dataset as an input (+ Add Input, right sidebar)?"
assert os.path.exists(val_file), f"Not found: {val_file}"
print("Train file:", train_file, "--", sum(1 for _ in open(train_file)), "rows")
print("Val file:", val_file, "--", sum(1 for _ in open(val_file)), "rows")

**Cross-session resume on Kaggle.** There's no Drive-style live mount here -- `/kaggle/working` only persists once you **Save Version** ("commit") the notebook, which turns its contents into this notebook's own Output, downloadable as a dataset. To continue training in a later session:

1. This session: train, then **Save Version** before your quota/time runs out. The committed `/kaggle/working/<output_dir_name>` becomes an Output you can download or directly reuse.
2. Next session: either (a) add *this same notebook's* previous Output version as an input (Kaggle lets you pick a specific version's output), or (b) download the `final`/`checkpoint-N` folder and re-upload it as its own small Dataset -- same idea as the Colab notebook's cross-account resume.
3. Point `resume_from_checkpoint_path` below at wherever that folder landed under `/kaggle/input/...`.

Leave `resume_from_checkpoint_path` blank for a first run.

In [ ]:
resume_from_checkpoint_path = ""  # @param {type:"string"}
# e.g. /kaggle/input/model1-ord150k-checkpoint/checkpoint-4750  (full Trainer checkpoint -- exact resume)
# or   /kaggle/input/model1-ord150k-checkpoint/final           (weights only -- fresh optimizer/step count)

In [ ]:
output_dir = "/kaggle/working/model1_reactant_ord150k"  # @param {type:"string"}
time_budget_minutes = 180  # @param {type:"number"}
# With torchrun --nproc_per_node=2 the effective batch size doubles (16/GPU x 2), so the
# ~9,188 steps/epoch this run needs (147k examples / 32) should finish well inside a single
# Kaggle session even accounting for DDP not being a perfect 2x -- 180 min (3h) leaves
# generous headroom. Lower this for a first smoke-test run.

In [ ]:
import os

os.makedirs(output_dir, exist_ok=True)
log_path = f"{output_dir}/train.log"
resume_flag = ["--resume-from-checkpoint", resume_from_checkpoint_path] if resume_from_checkpoint_path else []

!torchrun --nproc_per_node=2 scripts/train_reactant_model_ord.py \
    --train-file "{train_file}" \
    --val-file "{val_file}" \
    --output-dir "{output_dir}" \
    --local-work-dir /kaggle/temp/local_model1_work \
    --no-augment \
    --time-budget-minutes {time_budget_minutes} \
    {' '.join(resume_flag)} \
    > "{log_path}" 2>&1
print(f"Done (or paused at time budget). Log: {log_path}")

Same log-redirect reasoning as the Colab notebook: printing per-step output directly in the cell can make the tab unresponsive over a multi-hour run. Check progress by re-opening `train.log` from the Kaggle file browser on the left, or `!tail -40 {log_path}` in a scratch cell.

`--local-work-dir` points at `/kaggle/temp` (fast local scratch disk, wiped between sessions -- exactly the role `--local-work-dir` plays on Colab's `/content`) so Trainer's own checkpoint rotation never touches `/kaggle/working` directly; the training script's own `DriveSyncCallback`-style logic still copies out one `latest_checkpoint` folder under `output_dir` after every save (the name is a holdover from the Colab version -- functionally it's just "the one folder that survives a session restart," Drive or not). Only rank 0 (of the 2 `torchrun` processes) does this Drive-style sync and the final save, so there's no risk of the two GPU processes racing to write the same files.

**When done:** `output_dir/final` (or `output_dir/latest_checkpoint`) has the model. Evaluate it exactly like the other checkpoints, e.g. from a local shell after downloading:

```
python scripts/models/run_reactiont5_topk.py \
    --input data/v2_ord_eval_targets.json \
    --t5-model <downloaded_final_dir> \
    --num-beams 10 --output experiments/v2_model1_topk/ord150k_noaug_topk.json
```